# MPAS — reading a global model grid

**MPAS** (Model for Prediction Across Scales) is a weather/climate model whose grid is a mesh of **Voronoi
cells covering the whole globe**. This notebook reads one with the `mpas` driver and shows what is in it.

## 1. Read the file

In [ ]:
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from cleopatra.config import Config

from pyramids.netcdf.ugrid import MeshDataset

DATA = Path("../../../examples/data/real-data")
grid = DATA / "mpas" / "x1.1024002.grid.nc"

t0 = time.perf_counter()
ds = MeshDataset.read_file(grid, driver="mpas")
secs = time.perf_counter() - t0
Config.set_matplotlib_backend()

print(f"read in {secs:.2f} s")
print("nodes:", ds.n_node)
print("faces:", ds.n_face)

This grid has over **a million cells** covering the entire planet. One thing to know: MPAS stores its
coordinates in **radians**, not degrees — longitude runs `-π … π` and latitude `-π/2 … π/2`. We convert to
degrees before plotting.

In [ ]:
lon_rad = np.asarray(ds.mesh.node_x)
lat_rad = np.asarray(ds.mesh.node_y)
lon = np.degrees(lon_rad)
lat = np.degrees(lat_rad)

print("longitude: %.0f ... %.0f degrees" % (lon.min(), lon.max()))
print("latitude:  %.0f ... %.0f degrees" % (lat.min(), lat.max()))

## 2. The grid covers the whole globe

Each cell has a centre; plotting the node points (in degrees) shows how the mesh blankets the planet. There
are two million points, so we draw a sample of them, coloured by latitude.

In [ ]:
step = 40  # draw every 40th point
fig, ax = plt.subplots(figsize=(9, 4.5))
sc = ax.scatter(lon[::step], lat[::step], c=lat[::step], cmap="coolwarm", s=2, linewidths=0)
plt.colorbar(sc, label="latitude (degrees)")
ax.set_aspect("equal")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_title("MPAS grid — global coverage")

The points fill the whole `-180 … 180` longitude by `-90 … 90` latitude range evenly: this is a
**quasi-uniform** global grid, with cells of roughly equal size everywhere (that even spacing is the whole
point of an MPAS mesh).

## Takeaway

- The `mpas` driver reads a **million-cell** global grid into an ordinary pyramids `MeshDataset` in a couple
  of seconds.
- MPAS coordinates are in **radians** — convert with `np.degrees(...)` before plotting or comparing to
  lon/lat in degrees.
- The grid covers the entire globe with roughly uniform cell spacing; from there every pyramids operation
  works on it like any other mesh.